# Notebook 02 — Engenharia de Features

Este notebook carrega os dados limpos e aplica as transformações necessárias para a modelagem.

**Input:** `../data/data_cleaned.parquet`

**Outputs:**
- `../data/data_features.parquet` — features transformadas
- `../data/data_cleaned_with_ids.parquet` — dados originais com IDs para perfil estatístico

## Bibliotecas

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from scipy import stats
from sklearn.preprocessing import StandardScaler
from IPython.display import display

## Carregamento dos dados limpos

In [2]:
df_bis = pd.read_parquet('../data/data_cleaned.parquet')
print(f"Dados carregados: {df_bis.shape[0]} linhas x {df_bis.shape[1]} colunas")
display(df_bis.head())

Dados carregados: 2214 linhas x 13 colunas


,cod_pa,des_material,qt_altura_material,qt_comprimento_material,qt_peso_bruto,qt_volume_material,selagem_BOOM,selagem_MLUA,selagem_RETA,selagem_RETA_REC,selagem_TRANS,selagem_TRANS-REC,diametro_material
0,12504,CAPRICHO LOC HID DES CPO 200g,184.0,77.0,225.178,0.641,0,0,1,0,0,0,50.0
1,11881,ACQ FRESCA LOC HID 200g,182.0,80.0,227.014,0.641,0,0,1,0,0,0,50.0
2,10728,BOT MEN SHAMPOO ANTI-QUEDA 200ml,188.0,67.0,256.000,0.655,0,0,1,0,0,0,50.0
3,59147,LILY ESF HID CPO ACETINADO 200g,205.0,80.0,230.596,787.200,0,1,0,0,0,0,50.0
4,11790,VITACT PREENCHEDOR RUGAS 20g,147.0,30.0,37.050,182.700,0,0,1,0,0,0,19.0


## Aplicação da função Box-Cox

In [3]:
colunas_numericas = ['diametro_material', 'qt_altura_material', 'qt_comprimento_material', 'qt_peso_bruto']

df_numerico = df_bis[colunas_numericas].copy()

df_boxcox = pd.DataFrame()

for coluna in df_numerico.columns:
    # boxcox 
    dados_transformados, _ = stats.boxcox(df_numerico[coluna])
    
    df_boxcox[f'{coluna}_boxcox'] = dados_transformados
    
display(df_boxcox.head())

,diametro_material_boxcox,qt_altura_material_boxcox,qt_comprimento_material_boxcox,qt_peso_bruto_boxcox
0,17725.878568,5212.456509,2.046130,49.384130
1,17725.878568,5113.791751,2.052679,49.651419
2,17725.878568,5412.201410,2.021421,53.774691
3,17725.878568,6296.683084,2.052679,50.170693
4,1226.282796,3520.086905,1.848006,14.366283


### Parâmetros (Lambdas) Ótimos Usados na Transformação Box-Cox

In [4]:
colunas_numericas = ['diametro_material', 'qt_altura_material', 'qt_comprimento_material', 'qt_peso_bruto']

lambdas_encontrados = {}

for coluna in colunas_numericas:
    _, lambda_otimo = stats.boxcox(df_bis[coluna])
    lambdas_encontrados[coluna] = lambda_otimo


for coluna, lmbda in lambdas_encontrados.items():
    print(f" -> Para a coluna '{coluna}', o melhor lambda encontrado foi: {lmbda:.4f}")

 -> Para a coluna 'diametro_material', o melhor lambda encontrado foi: 2.7602
 -> Para a coluna 'qt_altura_material', o melhor lambda encontrado foi: 1.7484
 -> Para a coluna 'qt_comprimento_material', o melhor lambda encontrado foi: -0.4043
 -> Para a coluna 'qt_peso_bruto', o melhor lambda encontrado foi: 0.6445


## Normalização dos dados — StandardScaler

In [5]:
#std
scaler = StandardScaler()

# StandardScaler
dt_prep_array = scaler.fit_transform(df_boxcox)

df_preparado = pd.DataFrame(dt_prep_array, columns=df_boxcox.columns)

display(df_preparado.head())
display (df_preparado.describe())

,diametro_material_boxcox,qt_altura_material_boxcox,qt_comprimento_material_boxcox,qt_peso_bruto_boxcox
0,0.954094,0.851598,1.149530,0.911524
1,0.954094,0.785195,1.245517,0.930830
2,0.954094,0.986029,0.787370,1.228659
3,0.954094,1.581298,1.245517,0.968338
4,-1.965341,-0.287391,-1.754357,-1.617855


,diametro_material_boxcox,qt_altura_material_boxcox,qt_comprimento_material_boxcox,qt_peso_bruto_boxcox
count,2.214000e+03,2.214000e+03,2.214000e+03,2.214000e+03
mean,-7.702360e-17,4.236298e-16,4.454532e-15,-1.668845e-16
std,1.000226e+00,1.000226e+00,1.000226e+00,1.000226e+00
min,-1.965341e+00,-2.596581e+00,-3.708216e+00,-2.211657e+00
25%,-1.010524e+00,-8.219830e-01,-5.130891e-01,-8.550273e-01
50%,-4.882512e-01,-5.736971e-02,-4.442683e-02,-2.192014e-02
75%,9.540939e-01,7.193363e-01,5.708355e-01,9.126611e-01
max,9.540939e-01,3.131045e+00,3.276291e+00,1.846796e+00


## Merge dos datasets (features escaladas + selagem OHE)

In [6]:
print("Colunas preparadas:", df_preparado.columns.tolist())
print("Colunas originais:", df_bis.columns.tolist())

Colunas preparadas: ['diametro_material_boxcox', 'qt_altura_material_boxcox', 'qt_comprimento_material_boxcox', 'qt_peso_bruto_boxcox']
Colunas originais: ['cod_pa', 'des_material', 'qt_altura_material', 'qt_comprimento_material', 'qt_peso_bruto', 'qt_volume_material', 'selagem_BOOM', 'selagem_MLUA', 'selagem_RETA', 'selagem_RETA_REC', 'selagem_TRANS', 'selagem_TRANS-REC', 'diametro_material']


In [7]:
#'df_bis' + 'df_preparado' 
colunas_selagem = [col for col in df_bis.columns if col.startswith('selagem_')]
df_selagem_ohe = df_bis[colunas_selagem]

# merge dos df's
df_preparado.reset_index(drop=True, inplace=True)
df_selagem_ohe.reset_index(drop=True, inplace=True)

df_model= pd.concat([df_preparado, df_selagem_ohe], axis=1)

display(df_model.head())

,diametro_material_boxcox,qt_altura_material_boxcox,qt_comprimento_material_boxcox,qt_peso_bruto_boxcox,selagem_BOOM,selagem_MLUA,selagem_RETA,selagem_RETA_REC,selagem_TRANS,selagem_TRANS-REC
0,0.954094,0.851598,1.149530,0.911524,0,0,1,0,0,0
1,0.954094,0.785195,1.245517,0.930830,0,0,1,0,0,0
2,0.954094,0.986029,0.787370,1.228659,0,0,1,0,0,0
3,0.954094,1.581298,1.245517,0.968338,0,1,0,0,0,0
4,-1.965341,-0.287391,-1.754357,-1.617855,0,0,1,0,0,0


## Salvamento dos dados de features

In [ ]:
# Salva as features transformadas para modelagem
df_model.to_parquet('../data/data_features.parquet', index=False)
print(f"✓ Features salvas em '../data/data_features.parquet'")
print(f"  Shape: {df_model.shape}")
print(f"  Colunas: {df_model.columns.tolist()}")

# Salva os dados limpos 
df_bis.reset_index(drop=True, inplace=True)
df_bis.to_parquet('../data/data_cleaned_with_ids.parquet', index=False)
print(f"\n✓ Dados limpos com IDs salvos em '../data/data_cleaned_with_ids.parquet'")
print(f"  Shape: {df_bis.shape}")

✓ Features salvas em '../data/data_features.parquet'
  Shape: (2214, 10)
  Colunas: ['diametro_material_boxcox', 'qt_altura_material_boxcox', 'qt_comprimento_material_boxcox', 'qt_peso_bruto_boxcox', 'selagem_BOOM', 'selagem_MLUA', 'selagem_RETA', 'selagem_RETA_REC', 'selagem_TRANS', 'selagem_TRANS-REC']

✓ Dados limpos com IDs salvos em '../data/data_cleaned_with_ids.parquet'
  Shape: (2214, 13)
